In [1]:
import cv2
import mediapipe as mp
import numpy as np
import os

# --- Drawing Utilities Setup ---
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

RAW_DATA_PATH = os.path.join('../data', 'raw_videos')
EXPORT_DATA_PATH = os.path.join('../data', 'extracted_landmarks')
actions = ['hello', 'help', 'please', 'sorry', 'thank you'] 
TARGET_FRAMES = 30

try:
    mp_holistic = mp.solutions.holistic
except AttributeError:
    from mediapipe.python import solutions as mp_solutions
    mp_holistic = mp_solutions.holistic

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(63)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(63)
    return np.concatenate([pose, lh, rh])

def process_dataset():
    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        for action in actions:
            action_path = os.path.join(RAW_DATA_PATH, action)
            export_dir = os.path.join(EXPORT_DATA_PATH, action)
            
            if not os.path.exists(action_path):
                print(f"Directory {action_path} not found. Skipping...")
                continue
                
            os.makedirs(export_dir, exist_ok=True)
            videos = os.listdir(action_path)
            
            for video_idx, video_name in enumerate(videos):
                video_path = os.path.join(action_path, video_name)
                cap = cv2.VideoCapture(video_path)
                video_keypoints = []
                
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret:
                        break
                    
                    # 1. Image Processing
                    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    image.flags.writeable = False
                    results = holistic.process(image)
                    
                    # 2. Draw the "Thinking" (Landmarks) onto the frame
                    image.flags.writeable = True
                    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # Convert back for CV2 display
                    
                    # Draw Pose
                    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                                             mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                                             mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1))
                    # Draw Left Hand
                    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
                    # Draw Right Hand
                    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

                    # 3. Show the visualization window
                    cv2.imshow('MediaPipe Holistic Data Extraction', image)
                    
                    # 4. Extract data
                    keypoints = extract_keypoints(results)
                    video_keypoints.append(keypoints)

                    # Allow quitting with 'q'
                    if cv2.waitKey(1) & 0xFF == ord('q'):
                        cap.release()
                        cv2.destroyAllWindows()
                        return

                cap.release()
                
                # Standardize to 30 frames
                current_length = len(video_keypoints)
                if current_length > TARGET_FRAMES:
                    video_keypoints = video_keypoints[:TARGET_FRAMES]
                else:
                    padding = TARGET_FRAMES - current_length
                    for _ in range(padding):
                        video_keypoints.append(np.zeros(258))
                
                final_array = np.array(video_keypoints)
                np.save(os.path.join(export_dir, f"{video_idx}.npy"), final_array)
                print(f"Processed {action}/{video_name}")

        cv2.destroyAllWindows()

process_dataset()

I0000 00:00:1775213930.807332   46407 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775213930.810952   46491 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.3.6), renderer: AMD Radeon RX 6700 XT (radeonsi, navi22, LLVM 21.1.8, DRM 3.64, 6.19.10-200.fc43.x86_64)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
QFontDatabase: Cannot find font directory /home/huy/Code/Projects/DATN/DATN_SignLanguageDetection/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/huy/Code/Projects/DATN/DATN_SignLanguageDetection/.venv/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/huy/Code/Projects/DATN/DATN_SignLanguageDete

Processed hello/hello_0_3287.webm
Processed hello/hello_177_4088.mp4
Processed hello/hello_214_4997.webm
Processed hello/hello_152_7146.mp4
Processed hello/hello_262_7768.mp4
Processed hello/hello_77_10525.mp4
Processed hello/hello_2_11690.mp4
Processed hello/hello_243_11981.webm
Processed hello/hello_124_12586.mp4
Processed hello/hello_12_12661.webm
Processed hello/hello_330_12947.mp4
Processed hello/hello_96_12984.mp4
Processed hello/hello_394_13413.mp4
Processed hello/hello_72_13982.mp4
Processed hello/hello_160_14002.mp4
Processed hello/hello_160_14003.mp4
Processed help/help_77_1604.mp4
Processed help/help_77_1605.mp4
Processed help/help_2_1752.mp4
Processed help/help_0_3297.mp4
Processed help/help_144_4035.mp4
Processed help/help_144_4036.mp4
Processed help/help_94_4326.mp4
Processed help/help_205_4938.mp4
Processed help/help_205_4939.mp4
Processed help/help_205_4940.mp4
Processed help/help_233_6716.mp4
Processed help/help_152_7723.mp4
Processed help/help_0_8480.mp4
Processed hel